In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
import numpy as np
import os
import gc

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, recall_score

In [7]:
BASE_PATH = "/content/drive/MyDrive"

TRAIN_PATH = f"{BASE_PATH}/MajorProject/CIC_IoT_2023/Full_Dataset/train_working.csv"
TEST_PATH = f"{BASE_PATH}/MajorProject/CIC_IoT_2023/Full_Dataset/test_frozen.csv"

LABEL_COL = "label"
RANDOM_SEED = 42

In [8]:
print("Loading Training data")
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)

print("Loading Test data")
test_df = pd.read_csv(TEST_PATH, low_memory=False)

print("Datasets Loaded Succesfully")

Loading Training data
Loading Test data
Datasets Loaded Succesfully


In [9]:
train_df.sample(10)

,flow_duration,Header_Length,Protocol Type,Duration,Rate,Srate,Drate,fin_flag_number,syn_flag_number,rst_flag_number,...,Std,Tot size,IAT,Number,Magnitue,Radius,Covariance,Variance,Weight,label
30711,3.090114,79.92,6.00,64.00,0.310683,0.310683,0.0,0.0,1.0,0.0,...,0.000000,54.00,8.309013e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DDoS-SYN_Flood
222373,5.008240,110.88,6.11,63.80,0.439084,0.439084,0.0,0.0,0.0,0.0,...,0.000000,54.00,8.295083e+07,9.5,10.392305,0.000000,0.000000,0.00,141.55,DoS-TCP_Flood
696960,0.039423,77.20,6.00,58.70,23.286267,23.286267,0.0,0.0,0.0,1.0,...,1.074333,55.20,2.457809e-03,5.5,10.457480,1.519337,1.929642,0.60,38.50,Recon-HostDiscovery
672480,833.405162,2096.80,7.10,66.40,21.560815,21.560815,0.0,0.0,0.0,0.0,...,85.670217,144.80,1.664820e+08,13.5,15.809205,121.379368,7681.694413,1.00,244.60,SqlInjection
626733,700.999169,42272.20,13.70,156.70,0.378119,0.378119,0.0,0.0,0.0,0.0,...,71.590423,174.40,9.774680e-02,5.5,19.529993,101.244147,5800.220032,0.90,38.50,Recon-HostDiscovery
781668,4.535697,2338.90,6.60,64.89,2.628212,2.628212,0.0,0.0,0.0,0.0,...,314.998194,225.71,8.315819e+07,9.5,21.798380,446.072732,190999.496594,0.93,141.55,DDoS-HTTP_Flood
464059,2.406991,118318.58,16.45,67.24,33.289423,33.289423,0.0,0.0,0.0,0.0,...,546.960758,876.80,8.337100e+07,9.5,40.881833,773.474591,315012.336490,0.95,141.55,DDoS-UDP_Fragmentation
14135,0.068803,13815.00,17.00,64.00,11899.506987,11899.506987,0.0,0.0,0.0,0.0,...,0.000000,50.00,8.310223e+07,9.5,10.000000,0.000000,0.000000,0.00,141.55,DDoS-UDP_Flood
890326,31.198222,318981.50,14.20,64.20,38.961471,38.961471,0.0,0.0,0.0,0.0,...,258.102647,191.70,5.388403e-03,5.5,19.763573,365.012264,74841.328320,0.90,38.50,DNS_Spoofing
465787,0.000000,741.86,5.94,63.36,11.392302,11.392302,0.0,0.0,0.0,0.0,...,540.822114,929.46,8.334058e+07,9.5,43.452786,764.845340,308001.157304,0.95,141.55,DDoS-ACK_Fragmentation


In [10]:
print(f"Shape of training data: {train_df.shape}")
print(f"Shape of test data: {test_df.shape}")

Shape of training data: (900000, 47)
Shape of test data: (259646, 47)


### Clean data

In [11]:
train_df = train_df.dropna(subset=[LABEL_COL])
test_df = test_df.dropna(subset=[LABEL_COL])

print("Train shape after cleaning", train_df.shape)
print("Test shape after cleaning", test_df.shape)

Train shape after cleaning (900000, 47)
Test shape after cleaning (259645, 47)


### Split features and target

In [12]:
x_train = train_df.drop(columns=[LABEL_COL])
y_train = train_df[LABEL_COL]

x_test = test_df.drop(columns=[LABEL_COL])
y_test = test_df[LABEL_COL]

del train_df, test_df
gc.collect()

0

In [13]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 900000 entries, 0 to 899999
Data columns (total 46 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   flow_duration    900000 non-null  float64
 1   Header_Length    900000 non-null  float64
 2   Protocol Type    900000 non-null  float64
 3   Duration         900000 non-null  float64
 4   Rate             900000 non-null  float64
 5   Srate            900000 non-null  float64
 6   Drate            900000 non-null  float64
 7   fin_flag_number  900000 non-null  float64
 8   syn_flag_number  900000 non-null  float64
 9   rst_flag_number  900000 non-null  float64
 10  psh_flag_number  900000 non-null  float64
 11  ack_flag_number  900000 non-null  float64
 12  ece_flag_number  900000 non-null  float64
 13  cwr_flag_number  900000 non-null  float64
 14  ack_count        900000 non-null  float64
 15  syn_count        900000 non-null  float64
 16  fin_count        900000 non-null  floa

In [14]:
print("NANs in x_train:", x_train.isna().sum().sum())
print("NaNs in X_test:", x_test.isna().sum().sum())
print("NaNs in y_train:", y_train.isna().sum())
print("NaNs in y_test:", y_test.isna().sum())

NANs in x_train: 0
NaNs in X_test: 0
NaNs in y_train: 0
NaNs in y_test: 0


### Encode labels

In [15]:
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

class_names = le.classes_

print("Number of classes", len(class_names))

Number of classes 34


### Define Random Forest

In [16]:
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbose=1
)

### Train the model

In [17]:
print("Training Random forest on", x_train.shape[0], "rows")

rf_model.fit(x_train, y_train_enc)

Training Random forest on 900000 rows


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  3.4min
[Parallel(n_jobs=-1)]: Done 150 out of 150 | elapsed: 11.1min finished


RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42, verbose=1)

In [18]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    4.0s
[Parallel(n_jobs=2)]: Done 150 out of 150 | elapsed:   15.0s finished


In [19]:
print("Generating predictions...")
y_pred_enc = rf_model.predict(x_test)

Generating predictions...


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:    3.7s
[Parallel(n_jobs=2)]: Done 150 out of 150 | elapsed:   13.6s finished


In [20]:
report = classification_report(
    y_test_enc,
    y_pred_enc,
    target_names=class_names,
    digits=4
)

print(report)

                         precision    recall  f1-score   support

       Backdoor_Malware     0.9605    0.4953    0.6536       638
          BenignTraffic     0.7576    0.7472    0.7524      6000
       BrowserHijacking     0.9700    0.4439    0.6091      1167
       CommandInjection     0.9343    0.6067    0.7357      1078
 DDoS-ACK_Fragmentation     0.9976    0.9956    0.9966      7799
        DDoS-HTTP_Flood     0.9965    0.9920    0.9942      5742
        DDoS-ICMP_Flood     1.0000    0.9990    0.9995      6000
DDoS-ICMP_Fragmentation     0.9979    0.9957    0.9968      7655
      DDoS-PSHACK_Flood     0.9997    0.9998    0.9998      6000
       DDoS-RSTFINFlood     0.9998    0.9993    0.9996      6000
         DDoS-SYN_Flood     0.9983    0.9975    0.9979      6000
         DDoS-SlowLoris     0.9813    0.9970    0.9891      4672
DDoS-SynonymousIP_Flood     0.9993    0.9978    0.9986      6000
         DDoS-TCP_Flood     0.9998    0.9983    0.9991      6000
         DDoS-UDP_Flood 

In [21]:
recalls = recall_score(
    y_test_enc,
    y_pred_enc,
    average=None
)

recall_df = pd.DataFrame({
    "Class": class_names,
    "Recall": recalls
}).sort_values("Recall")

recall_df

,Class,Recall
28,Recon-PingSweep,0.151786
31,Uploading_Attack,0.292000
30,SqlInjection,0.393678
2,BrowserHijacking,0.443873
0,Backdoor_Malware,0.495298
33,XSS,0.511749
17,DictionaryBruteForce,0.583237
3,CommandInjection,0.606679
1,BenignTraffic,0.747167
29,Recon-PortScan,0.749161


In [ ]:
MODEL_A_OUTPUT_DIR = f"{BASE_PATH}/MajorProject/IDS_ModelA_Outputs"
os.makedirs(MODEL_A_OUTPUT_DIR, exist_ok=True)

recall_path = f"{MODEL_A_OUTPUT_DIR}/model_A_per_class_recall.csv"
recall_df.to_csv(recall_path, index=False)

print("✅ Per-class recall saved to:", recall_path)

✅ Per-class recall saved to: /content/drive/MyDrive/IDS_ModelA_Outputs/model_A_per_class_recall.csv


In [ ]:
import joblib

joblib.dump(rf_model, f"{MODEL_A_OUTPUT_DIR}/modelA_RF.joblib")
joblib.dump(le, f"{MODEL_A_OUTPUT_DIR}/modelA_labelEncoder.joblib")

print("Model A Saved succesfully")

Model A Saved succesfully


: 